# Step 3 — Week 3: Retrieval + Generation Chain

## Goals
- Build retrieval + generation chain
- Add metadata filters, top-k, and score thresholds
- Implement citation-style output (source chunks)

## Prerequisites
- Ollama installed and running: `ollama serve`
- Model pulled: `ollama pull llama3.2`
- Dependencies: `pip install requests scikit-learn pandas`

In [1]:
import re
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.2"


def ollama_generate(prompt: str, system: str = "You are a helpful assistant.", temperature: float = 0.2) -> str:
    body = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "top_p": 0.9},
        "system": system,
    }
    response = requests.post(f"{OLLAMA_URL}/api/generate", json=body, timeout=120)
    response.raise_for_status()
    return response.json()["response"].strip()

print("Setup complete")

Setup complete


## 1) Build Knowledge Chunks with Metadata

In [2]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    topic: str
    source: str
    text: str


CHUNKS: List[Chunk] = [
    Chunk("c1", "doc_rag_1", "rag", "RAG Guide", "RAG combines retrieval with generation to ground answers in external knowledge."),
    Chunk("c2", "doc_rag_1", "rag", "RAG Guide", "Top-k controls how many chunks are passed to the generator, affecting recall and noise."),
    Chunk("c3", "doc_eval_1", "evaluation", "Eval Notes", "Faithfulness measures whether claims are supported by retrieved evidence."),
    Chunk("c4", "doc_eval_1", "evaluation", "Eval Notes", "Format compliance checks whether outputs match required structure."),
    Chunk("c5", "doc_db_1", "vector-db", "Vector DB Notes", "Metadata filters can restrict retrieval to selected subsets such as topic or source."),
    Chunk("c6", "doc_db_1", "vector-db", "Vector DB Notes", "Similarity thresholds can remove weak matches before generation."),
    Chunk("c7", "doc_chunk_1", "chunking", "Chunking Notes", "Overlap preserves context continuity across boundaries but increases index size."),
    Chunk("c8", "doc_chunk_1", "chunking", "Chunking Notes", "Semantic chunking often improves coherence by splitting on meaning shifts."),
]

print(f"Loaded {len(CHUNKS)} chunks")

Loaded 8 chunks


## 2) Retrieval Chain (Metadata Filters, Top-k, Score Threshold)

In [3]:
def retrieve(
    query: str,
    chunks: List[Chunk],
    top_k: int = 3,
    score_threshold: float = 0.10,
    metadata_filters: Optional[Dict[str, str]] = None,
) -> List[Dict[str, Any]]:
    metadata_filters = metadata_filters or {}

    # Apply metadata filters first
    filtered = []
    for chunk in chunks:
        ok = True
        for key, value in metadata_filters.items():
            if not hasattr(chunk, key) or str(getattr(chunk, key)) != str(value):
                ok = False
                break
        if ok:
            filtered.append(chunk)

    if not filtered:
        return []

    texts = [chunk.text for chunk in filtered]
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(texts)
    q_vec = vectorizer.transform([query])

    sims = cosine_similarity(q_vec, matrix).flatten()
    ranked_idx = np.argsort(-sims)

    results = []
    for idx in ranked_idx:
        score = float(sims[idx])
        if score < score_threshold:
            continue
        chunk = filtered[idx]
        results.append(
            {
                "chunk_id": chunk.chunk_id,
                "doc_id": chunk.doc_id,
                "topic": chunk.topic,
                "source": chunk.source,
                "text": chunk.text,
                "score": round(score, 4),
            }
        )
        if len(results) >= top_k:
            break

    return results


query = "How do metadata filters help retrieval?"
preview = retrieve(
    query=query,
    chunks=CHUNKS,
    top_k=3,
    score_threshold=0.05,
    metadata_filters={"topic": "vector-db"},
)

pd.DataFrame(preview)

,chunk_id,doc_id,topic,source,text,score
0,c5,doc_db_1,vector-db,Vector DB Notes,Metadata filters can restrict retrieval to sel...,0.5164


## 3) Generation Chain + Citation-Style Output

In [4]:
def format_context(retrieved: List[Dict[str, Any]]) -> str:
    blocks = []
    for r in retrieved:
        blocks.append(
            f"[{r['chunk_id']}] (source={r['source']}, topic={r['topic']}, score={r['score']})\n{r['text']}"
        )
    return "\n\n".join(blocks)


def rag_answer(
    question: str,
    top_k: int = 3,
    score_threshold: float = 0.10,
    metadata_filters: Optional[Dict[str, str]] = None,
) -> Dict[str, Any]:
    retrieved = retrieve(
        query=question,
        chunks=CHUNKS,
        top_k=top_k,
        score_threshold=score_threshold,
        metadata_filters=metadata_filters,
    )

    if not retrieved:
        return {
            "question": question,
            "answer": "I couldn't find relevant context with the current filters/threshold.",
            "citations": [],
            "retrieved": [],
        }

    context_text = format_context(retrieved)

    prompt = f"""Answer the question using only the provided context.
If information is missing, say so.

Question:
{question}

Context:
{context_text}

Output format:
1) A short answer paragraph
2) A line: Citations: [chunk_id1], [chunk_id2], ...
"""

    answer = ollama_generate(prompt, temperature=0.1)
    citations = [r["chunk_id"] for r in retrieved]

    return {
        "question": question,
        "answer": answer,
        "citations": citations,
        "retrieved": retrieved,
    }


example = rag_answer(
    question="How do top-k and score thresholds affect retrieval quality?",
    top_k=3,
    score_threshold=0.06,
    metadata_filters={"topic": "rag"},
)

print("Question:", example["question"])
print("\nAnswer:\n", example["answer"])
print("\nCitation chunk IDs:", example["citations"])
pd.DataFrame(example["retrieved"])[["chunk_id", "topic", "score", "text"]]

Question: How do top-k and score thresholds affect retrieval quality?

Answer:
 Top-k and score thresholds can significantly affect the retrieval quality of a system. Top-k refers to the number of top-ranked documents or answers retrieved by the system, while score thresholds determine the minimum confidence level required for an answer to be considered valid.

A higher top-k value means that more relevant answers are retrieved, but may also increase the noise and irrelevant information in the output. On the other hand, a lower top-k value can lead to fewer relevant answers being retrieved, but with less noise.

Score thresholds work similarly, where a higher threshold requires stronger evidence or confidence for an answer to be accepted. A lower score threshold allows for weaker evidence, which may increase retrieval quality by including more possible answers, but also increases the risk of retrieving incorrect information.

The optimal top-k and score thresholds depend on the specifi

,chunk_id,topic,score,text
0,c1,rag,0.2582,RAG combines retrieval with generation to grou...


In [5]:
# Multi-query demo for Week 3 objectives
cases = [
    {
        "question": "What is faithfulness in RAG evaluation?",
        "top_k": 2,
        "score_threshold": 0.05,
        "metadata_filters": {"topic": "evaluation"},
    },
    {
        "question": "Why use metadata filters in retrieval?",
        "top_k": 3,
        "score_threshold": 0.05,
        "metadata_filters": {"topic": "vector-db"},
    },
    {
        "question": "How does overlap affect chunking?",
        "top_k": 3,
        "score_threshold": 0.05,
        "metadata_filters": {"topic": "chunking"},
    },
]

for idx, case in enumerate(cases, start=1):
    out = rag_answer(**case)
    print("\n" + "=" * 80)
    print(f"Case {idx}: {case['question']}")
    print(f"Filters={case['metadata_filters']} | top_k={case['top_k']} | threshold={case['score_threshold']}")
    print("Citations:", out["citations"])
    print("Answer preview:", out["answer"][:240].replace("\n", " "), "...")


Case 1: What is faithfulness in RAG evaluation?
Filters={'topic': 'evaluation'} | top_k=2 | threshold=0.05
Citations: ['c3']
Answer preview: Faithfulness in RAG (Relevance and Accuracy of Generated) evaluation assesses the extent to which a generated text's claims are supported by retrieved evidence. In other words, it evaluates how well the model has linked its generated conten ...

Case 2: Why use metadata filters in retrieval?
Filters={'topic': 'vector-db'} | top_k=3 | threshold=0.05
Citations: ['c5']
Answer preview: Metadata filters are used in retrieval to narrow down the search results by specifying certain criteria, such as topic or source. This helps to restrict the output to a more focused and relevant set of information.  Citations: [c5] ...

Case 3: How does overlap affect chunking?
Filters={'topic': 'chunking'} | top_k=3 | threshold=0.05
Citations: ['c8', 'c7']
Answer preview: Overlap in chunking refers to the preservation of contextual information across boundaries, which